In [0]:
import subprocess
from databricks.sdk import WorkspaceClient

REPO_NAME = "git-test"
DEFAULT_BRANCH = "main"

me = WorkspaceClient().current_user.me().user_name
repo = f"/Workspace/Users/{me}/{REPO_NAME}"

def git(*args):
    r = subprocess.run(["git", "-C", repo, *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stderr}")
    return r.stdout.strip()

# Refresh what exists on GitHub and forget remote branches that were deleted
git("fetch", "--prune", "origin")

# Local branches whose GitHub copy is gone
lines = git("for-each-ref", "--format=%(refname:short)|%(upstream:track)", "refs/heads").splitlines()
gone = [l.split("|")[0] for l in lines if "[gone]" in l]

if not gone:
    print("Nothing to clean up. You can create your branch from main.")
else:
    current = git("rev-parse", "--abbrev-ref", "HEAD")
    if current in gone:
        if git("status", "--porcelain"):
            raise RuntimeError("You have unsaved changes on a deleted branch. "
                               "Ask the dev team before continuing.")
        git("checkout", DEFAULT_BRANCH)
        git("pull", "origin", DEFAULT_BRANCH)
    for b in gone:
        git("branch", "-D", b)
        print(f"Removed old branch: {b}")
    print("Done. You can now create your branch from main.")

In [0]:
import os, subprocess
print("Repo path exists:", os.path.exists(repo))
print(".git exists:", os.path.exists(f"{repo}/.git"))
print(".git/config readable:", os.access(f"{repo}/.git/config", os.R_OK))
print(subprocess.run(["git", "--version"], capture_output=True, text=True).stdout)